# 🌋 Analisis Dispersi Abu Vulkanik dengan HYSPLIT

Notebook ini berisi 4 langkah utama untuk menganalisis penyebaran abu vulkanik:
1. **Konfigurasi Parameter** - Atur parameter simulasi
2. **Jalankan Simulasi** - HYSPLIT dispersion modeling
3. **Konversi & Analisis** - Extract metrik spasial
4. **Export Hasil** - Simpan ke CSV/Excel

---

## ⚙️ PARAMETER KONFIGURASI

**Edit parameter di bawah ini sesuai kebutuhan, lalu jalankan semua cell secara berurutan:**

In [13]:
# ========================================================================
# PARAMETER KONFIGURASI SIMULASI
# ========================================================================
# Edit parameter di bawah ini, lalu jalankan cell ini sebelum Step 1
# Semua parameter ini akan digunakan untuk membuat file CONTROL HYSPLIT

# 📍 LOKASI ERUPSI
LAT_SUMBER = -8.108        # Latitude sumber erupsi (contoh: Gunung Semeru)
LON_SUMBER = 112.922       # Longitude sumber erupsi

# 🌋 KARAKTERISTIK ERUPSI
TINGGI_LETUSAN_M = 3000    # Tinggi kolom abu dari permukaan laut (meter)
                           # - Erupsi kecil: 1000-3000 m
                           # - Erupsi sedang: 3000-8000 m
                           # - Erupsi besar: >8000 m

EMISSION_RATE = 1000.0     # Laju emisi abu (unit/jam)
                           # - Kecil: 100-1000
                           # - Sedang: 1000-5000
                           # - Besar: >5000

# ⚛️ PROPERTI PARTIKEL (GRAVITASI & SETTLING)
# Parameter ini menentukan kecepatan jatuh partikel akibat gravitasi Bumi (9,807 m/s²)
PARTICLE_DIAMETER_UM = 30.0   # Diameter partikel abu (mikrometer)
                              # - Fine ash: 10-30 μm
                              # - Medium ash: 30-100 μm
                              # - Coarse ash: 100-1000 μm

PARTICLE_DENSITY_G_CM3 = 2.5  # Densitas partikel (g/cm³)
                              # - Basaltik: 2.7-3.0 g/cm³
                              # - Andesitik: 2.5-2.7 g/cm³
                              # - Pumice: 0.5-1.5 g/cm³

PARTICLE_SHAPE = 0.7          # Shape factor (0=flat, 1=spherical)
                              # - Irregular ash: 0.5-0.7
                              # - Sub-rounded: 0.7-0.9
                              # - Spherical: 0.9-1.0

# ⬇️ DEPOSISI & REMOVAL
DRY_DEPOSITION_VEL = 0.01     # Kecepatan deposisi kering (cm/s)
                              # - 0.0 = tidak ada deposisi
                              # - 0.01 = deposisi lambat
                              # - 0.1 = deposisi sedang
                              # - 1.0 = deposisi cepat

WET_REMOVAL_COEF = 0.0        # Koefisien wet removal (1/s)
                              # - 0.0 = tidak ada wet scavenging
                              # - 8.0E-5 = removal sedang oleh hujan

# ⏱️ DURASI SIMULASI
RUNTIME_JAM = 5            # Durasi simulasi dalam jam
                           # - Cepat (lokal): 3-6 jam
                           # - Sedang (regional): 12-24 jam
                           # - Lama (dispersi penuh): 48-72 jam

# 📅 WAKTU ERUPSI
TIMESTAMP = "25 03 01 00"  # Format: YY MM DD HH
                           # Contoh: "25 03 01 00" = 1 Maret 2025, 00:00
                           # ⚠️ HARUS sesuai dengan data meteorologi yang tersedia!

# 📂 FILE METEOROLOGI
MET_FILE = "gdas1.mar25.w1"  # Nama file data meteorologi GDAS
                             # Format: gdas1.MMMYY.wN
                             # ⚠️ Pastikan file ada di F:/HYSPLIT/metdata/

# 🗺️ RESOLUSI GRID
GRID_SPACING = 0.05        # Resolusi grid dalam derajat
                           # - 0.05° ≈ 5.5 km (presisi tinggi, lambat)
                           # - 0.10° ≈ 11 km (seimbang)
                           # - 0.25° ≈ 28 km (cepat, kasar)

GRID_SPAN_LAT = 30.0       # Jangkauan grid latitude (±derajat dari sumber)
GRID_SPAN_LON = 30.0       # Jangkauan grid longitude (±derajat dari sumber)
                           # 30.0 = mencakup area ±15° dari titik erupsi

# 🎯 THRESHOLD KONSENTRASI
THRESHOLD_CONC = 1e-10     # Konsentrasi minimum untuk dianggap terdampak (mg/m³)
                           # - 1e-12: Sangat sensitif (trace amounts)
                           # - 1e-10: Standar
                           # - 1e-8: Konservatif (hanya konsentrasi signifikan)

# 🔧 PARAMETER LANJUTAN (opsional)
TOP_OF_MODEL_M = 3000.0    # Ketinggian maksimum model atmosfer (meter)
VERTICAL_MOTION = 0        # 0=data, 1=isobaric, 2=isentropic, 3=constant density, 4=isosigma

# ========================================================================
# ESTIMASI JARAK DISPERSI (berdasarkan asumsi kecepatan angin)
# ========================================================================
# ⚠️ CATATAN: Kecepatan dan arah angin SEBENARNYA ditentukan oleh data
#    meteorologi GDAS, BUKAN dari parameter manual. Estimasi di bawah ini
#    hanya untuk referensi perencanaan.

# Estimasi kecepatan angin rata-rata (km/jam) - HANYA UNTUK REFERENSI
KEC_ANGIN_ESTIMASI_KM_JAM = 6  # Angin normal: 10-20 km/jam
                                # Angin kencang: 30-50 km/jam
                                # Badai: >60 km/jam

# Arah angin dominan (derajat) - HANYA UNTUK REFERENSI
# 0° = Utara, 90° = Timur, 180° = Selatan, 270° = Barat
ARAH_ANGIN_ESTIMASI_DEG = 270   # Estimasi arah angin (dari arah ini)

# Estimasi jarak maksimum penyebaran
ESTIMASI_JARAK_KM = KEC_ANGIN_ESTIMASI_KM_JAM * RUNTIME_JAM

# ========================================================================
print("=" * 70)
print("🌋 KONFIGURASI SIMULASI DISPERSI ABU VULKANIK")
print("=" * 70)

print(f"\n📍 LOKASI SUMBER:")
print(f"   • Koordinat: {LAT_SUMBER}°N, {LON_SUMBER}°E")
print(f"   • Tinggi letusan: {TINGGI_LETUSAN_M:,} m")
print(f"   • Emission rate: {EMISSION_RATE} unit/jam")

print(f"\n⏱️  SIMULASI:")
print(f"   • Durasi: {RUNTIME_JAM} jam")
print(f"   • Waktu mulai: {TIMESTAMP}")
print(f"   • File meteorologi: {MET_FILE}")

print(f"\n🗺️  GRID & RESOLUSI:")
print(f"   • Spacing: {GRID_SPACING}° ≈ {GRID_SPACING * 111.32:.1f} km/cell")
print(f"   • Area coverage: ±{GRID_SPAN_LAT/2:.0f}° latitude × ±{GRID_SPAN_LON/2:.0f}° longitude")
print(f"   • Threshold konsentrasi: {THRESHOLD_CONC} mg/m³")

print(f"\n⚛️  PROPERTI PARTIKEL (Gravitasi Bumi = 9.807 m/s²):")
print(f"   • Diameter partikel: {PARTICLE_DIAMETER_UM:.1f} μm")
print(f"   • Densitas partikel: {PARTICLE_DENSITY_G_CM3:.2f} g/cm³")
print(f"   • Shape factor: {PARTICLE_SHAPE:.2f}")
print(f"   • Dry deposition velocity: {DRY_DEPOSITION_VEL:.3f} cm/s")
print(f"   • Wet removal: {WET_REMOVAL_COEF} 1/s")

print(f"\n💨 ESTIMASI (referensi, bukan input simulasi):")
print(f"   • Kecepatan angin: ~{KEC_ANGIN_ESTIMASI_KM_JAM} km/jam")
print(f"   • Arah angin: ~{ARAH_ANGIN_ESTIMASI_DEG}° (dari arah ini)")
print(f"   • Estimasi jarak: ~{ESTIMASI_JARAK_KM} km")

print(f"\n⚠️  Hasil simulasi akan menggunakan data angin aktual dari {MET_FILE}")
print("=" * 70)

🌋 KONFIGURASI SIMULASI DISPERSI ABU VULKANIK

📍 LOKASI SUMBER:
   • Koordinat: -8.108°N, 112.922°E
   • Tinggi letusan: 3,000 m
   • Emission rate: 1000.0 unit/jam

⏱️  SIMULASI:
   • Durasi: 5 jam
   • Waktu mulai: 25 03 01 00
   • File meteorologi: gdas1.mar25.w1

🗺️  GRID & RESOLUSI:
   • Spacing: 0.05° ≈ 5.6 km/cell
   • Area coverage: ±15° latitude × ±15° longitude
   • Threshold konsentrasi: 1e-10 mg/m³

⚛️  PROPERTI PARTIKEL (Gravitasi Bumi = 9.807 m/s²):
   • Diameter partikel: 30.0 μm
   • Densitas partikel: 2.50 g/cm³
   • Shape factor: 0.70
   • Dry deposition velocity: 0.010 cm/s
   • Wet removal: 0.0 1/s

💨 ESTIMASI (referensi, bukan input simulasi):
   • Kecepatan angin: ~6 km/jam
   • Arah angin: ~270° (dari arah ini)
   • Estimasi jarak: ~30 km

⚠️  Hasil simulasi akan menggunakan data angin aktual dari gdas1.mar25.w1


---
## Step 1: Jalankan Simulasi HYSPLIT

**Prerequisites:**
- Data meteorologi sudah tersedia di `F:/HYSPLIT/metdata/`
- Folder working sudah tersedia: `F:/HYSPLIT/working/`
- **Parameter konfigurasi di atas sudah diatur dan dijalankan**

**Output:** File `cdump` akan terbentuk di folder working

**Troubleshooting:**
- Jika file cdump berukuran kecil (<500 bytes), jalankan Step 1.5 untuk debug
- Pastikan tanggal simulasi sesuai dengan file meteorologi yang tersedia
- Jika error, cek file MESSAGE di folder working

In [14]:
import subprocess
import os

# KONFIGURASI PATH
hysplit_dir = "F:/HYSPLIT/exec/hycs_std.exe"
working_dir = "F:/HYSPLIT/working"
metdata_dir = "F:/HYSPLIT/metdata"

def create_control_file():
    """
    Membuat file CONTROL untuk simulasi HYSPLIT menggunakan parameter
    dari cell PARAMETER KONFIGURASI.
    
    Semua parameter diambil dari variabel global:
    - LAT_SUMBER, LON_SUMBER: Koordinat erupsi
    - TINGGI_LETUSAN_M: Tinggi kolom abu
    - TIMESTAMP: Waktu erupsi
    - MET_FILE: File meteorologi
    - RUNTIME_JAM: Durasi simulasi
    - EMISSION_RATE: Laju emisi
    - GRID_SPACING: Resolusi grid
    """
    os.makedirs(working_dir, exist_ok=True)
    
    # Validasi parameter konfigurasi tersedia
    required_vars = ['LAT_SUMBER', 'LON_SUMBER', 'TINGGI_LETUSAN_M', 'TIMESTAMP', 
                     'MET_FILE', 'RUNTIME_JAM', 'EMISSION_RATE', 'GRID_SPACING',
                     'GRID_SPAN_LAT', 'GRID_SPAN_LON', 'TOP_OF_MODEL_M', 'VERTICAL_MOTION']
    for var in required_vars:
        if var not in globals():
            raise RuntimeError(f"Parameter '{var}' belum didefinisikan. Jalankan cell PARAMETER KONFIGURASI terlebih dahulu.")
    
    # Validasi format timestamp
    ts_parts = TIMESTAMP.strip().split()
    if len(ts_parts) != 4 or not all(part.isdigit() for part in ts_parts):
        raise ValueError("TIMESTAMP harus format 'YY MM DD HH', contoh: '25 03 01 00'")
    
    control_path = os.path.join(working_dir, "CONTROL")
    conc_level = max(float(TINGGI_LETUSAN_M), 1000.0)
    
    # ========================================================================
    # FORMAT CONTROL - SEMUA DATA DARI PARAMETER KONFIGURASI
    # ========================================================================
    control_lines = [
        TIMESTAMP,                                    # Start time dari konfigurasi
        "1",                                          # Number of source locations
        f"{LAT_SUMBER} {LON_SUMBER} {TINGGI_LETUSAN_M}",  # Source location
        str(RUNTIME_JAM),                             # Run duration dari konfigurasi
        str(VERTICAL_MOTION),                         # Vertical motion method dari konfigurasi
        f"{TOP_OF_MODEL_M}",                          # Top of model dari konfigurasi
        "1",                                          # Number of met files
        f"{metdata_dir}/",                            # Met file directory
        MET_FILE,                                     # Met file dari konfigurasi
        "1",                                          # Number of pollutants
        "ASH",                                        # Pollutant name
        str(EMISSION_RATE),                           # Emission rate dari konfigurasi
        f"{RUNTIME_JAM}.0",                           # Emission duration = runtime
        "00 00 00 00 00",                             # Release start time (relative)
        "1",                                          # Number of concentration grids
        f"{LAT_SUMBER} {LON_SUMBER}",                 # Grid center = source location
        f"{GRID_SPACING} {GRID_SPACING}",             # Grid spacing dari konfigurasi
        f"{GRID_SPAN_LAT} {GRID_SPAN_LON}",           # Grid span dari konfigurasi
        "./",                                         # Output directory
        "cdump",                                      # Output file name
        "1",                                          # Number of vertical levels
        f"{conc_level:.1f}",                          # Concentration level (m)
        "00 00 00 00 00",                             # Sampling start
        "00 00 00 00 00",                             # Sampling stop
        "00 01 00",                                   # Sampling interval (hourly)
        "1",                                          # Number of depositing pollutants
        f"{PARTICLE_DIAMETER_UM} {PARTICLE_DENSITY_G_CM3} {PARTICLE_SHAPE}",  # Particle properties
        f"{DRY_DEPOSITION_VEL} 0.0 0.0 0.0 0.0",     # Dry deposition velocity (cm/s)
        f"{WET_REMOVAL_COEF} 0.0 0.0",               # Wet removal coefficient
        "0.0",                                        # Radioactive decay half-life
        "0.0"                                         # Resuspension factor
    ]
    
    with open(control_path, "w") as f:
        f.write("\n".join(control_lines) + "\n")
    
    # Print konfigurasi yang digunakan
    grid_km = GRID_SPACING * 111.32
    print(f"✓ File CONTROL dibuat dengan konfigurasi:")
    print(f"  • Lokasi: {LAT_SUMBER}°N, {LON_SUMBER}°E")
    print(f"  • Tanggal: {TIMESTAMP}")
    print(f"  • Meteorologi: {MET_FILE}")
    print(f"  • Runtime: {RUNTIME_JAM} jam")
    print(f"  • Tinggi letusan: {TINGGI_LETUSAN_M:,} m")
    print(f"  • Emission rate: {EMISSION_RATE} unit/jam")
    print(f"  • Grid spacing: {GRID_SPACING}° ≈ {grid_km:.1f} km/cell")
    print(f"  • Level konsentrasi: {conc_level:.1f} m")
    print(f"  • Partikel: {PARTICLE_DIAMETER_UM}μm, {PARTICLE_DENSITY_G_CM3}g/cm³, shape={PARTICLE_SHAPE}")
    print(f"  • Deposisi: dry={DRY_DEPOSITION_VEL}cm/s, wet={WET_REMOVAL_COEF}/s")

def print_message_tail(max_lines=40):
    """Tampilkan potongan akhir file MESSAGE jika ada."""
    message_path = os.path.join(working_dir, "MESSAGE")
    if not os.path.exists(message_path):
        print("   File MESSAGE tidak ditemukan.")
        return

    print("\n📄 Potongan akhir MESSAGE:")
    try:
        with open(message_path, "r") as f:
            lines = f.readlines()
        for line in lines[-max_lines:]:
            print("   " + line.rstrip())
    except Exception as e:
        print(f"   Gagal membaca MESSAGE: {e}")

def run_hysplit(met_file):
    """Menjalankan simulasi HYSPLIT dengan pre-check path penting."""
    met_path = os.path.join(metdata_dir, met_file)

    if not os.path.exists(hysplit_dir):
        print(f"\n❌ Executable HYSPLIT tidak ditemukan: {hysplit_dir}")
        return False
    if not os.path.exists(working_dir):
        print(f"\n❌ Working directory tidak ditemukan: {working_dir}")
        return False
    if not os.path.exists(os.path.join(working_dir, "CONTROL")):
        print("\n❌ File CONTROL tidak ditemukan di working directory")
        return False
    if not os.path.exists(met_path):
        print(f"\n❌ File meteorologi tidak ditemukan: {met_path}")
        print("   Pastikan nama file dan tanggal simulasi sesuai data meteo.")
        return False

    try:
        result = subprocess.run(
            [hysplit_dir],
            check=True,
            cwd=working_dir,
            capture_output=True,
            text=True
        )
        print("\n🎉 SIMULASI BERHASIL!")
        print(f"   Hasil tersimpan di: {working_dir}/cdump")
        if result.stdout.strip():
            print("\n📄 Stdout:")
            print(result.stdout.strip())
        return True
    except subprocess.CalledProcessError as e:
        print(f"\n❌ Error {e.returncode}: Simulasi gagal")
        if e.stdout and e.stdout.strip():
            print("\n📄 Stdout:")
            print(e.stdout.strip())
        if e.stderr and e.stderr.strip():
            print("\n📄 Stderr:")
            print(e.stderr.strip())
        print("   Periksa kembali file CONTROL, MESSAGE, dan data meteorologi")
        print_message_tail()
        return False
    except Exception as e:
        print(f"\n❌ Terjadi kesalahan: {e}")
        return False

# ========================================================================
# JALANKAN SIMULASI MENGGUNAKAN PARAMETER DARI CELL KONFIGURASI
# ========================================================================

print("=" * 70)
print("STEP 1: SIMULASI HYSPLIT")
print("=" * 70)

# Cek apakah parameter konfigurasi sudah didefinisikan
if 'LAT_SUMBER' not in locals():
    print("\n⚠️  BELUM MENJALANKAN CELL KONFIGURASI!")
    print("   Jalankan cell 'PARAMETER KONFIGURASI' terlebih dahulu.\n")
else:
    # Buat file CONTROL menggunakan parameter dari konfigurasi
    create_control_file()
    
    # Jalankan simulasi HYSPLIT
    success = run_hysplit(MET_FILE)
    
    if success:
        print("\n✅ Step 1 selesai! Lanjut ke Step 2.")

    else:        print("\n⚠️  Perbaiki error di atas sebelum lanjut ke step berikutnya.")

STEP 1: SIMULASI HYSPLIT
✓ File CONTROL dibuat dengan konfigurasi:
  • Lokasi: -8.108°N, 112.922°E
  • Tanggal: 25 03 01 00
  • Meteorologi: gdas1.mar25.w1
  • Runtime: 5 jam
  • Tinggi letusan: 3,000 m
  • Emission rate: 1000.0 unit/jam
  • Grid spacing: 0.05° ≈ 5.6 km/cell
  • Level konsentrasi: 3000.0 m
  • Partikel: 30.0μm, 2.5g/cm³, shape=0.7
  • Deposisi: dry=0.01cm/s, wet=0.0/s

🎉 SIMULASI BERHASIL!
   Hasil tersimpan di: F:/HYSPLIT/working/cdump

📄 Stdout:
HYSPLIT - Initialization 
 HYSPLIT version: hysplit.v5.4.2
 Last Changed Date: 2025-05-14
 Calculation Started ... please be patient
 Percent complete:  20.0
 Percent complete:  40.0
 Percent complete:  60.0
 Percent complete:  80.0
 Percent complete: 100.0
 Complete Hysplit

✅ Step 1 selesai! Lanjut ke Step 2.


---
## Step 1.5: [OPSIONAL] Debug Simulasi

Jika simulasi tidak menghasilkan output, gunakan cell ini untuk mengecek file MESSAGE dan melihat detail error dari HYSPLIT.

In [15]:
import os

# Cek file MESSAGE untuk debug
message_file = "F:/HYSPLIT/working/MESSAGE"
warning_file = "F:/HYSPLIT/working/WARNING"

print("=" * 70)
print("DEBUG: CEK HASIL SIMULASI")
print("=" * 70)

# Cek ukuran file cdump
cdump_file = "F:/HYSPLIT/working/cdump"
if os.path.exists(cdump_file):
    size = os.path.getsize(cdump_file)
    print(f"\n📂 File cdump: {size} bytes")
    if size < 500:
        print("   ⚠️ File cdump terlalu kecil - kemungkinan tidak ada data konsentrasi")
else:
    print("\n❌ File cdump tidak ditemukan!")

# Cek WARNING
if os.path.exists(warning_file):
    print(f"\n⚠️ File WARNING ditemukan:")
    with open(warning_file, 'r') as f:
        warnings = f.read()
        if warnings.strip():
            print(warnings)
        else:
            print("   (kosong)")

# Cek MESSAGE (100 baris terakhir)
if os.path.exists(message_file):
    print(f"\n📝 100 baris terakhir dari MESSAGE:\n")
    with open(message_file, 'r') as f:
        lines = f.readlines()
        for line in lines[-100:]:
            print(line.rstrip())
else:
    print("\n❌ File MESSAGE tidak ditemukan!")

print("\n" + "=" * 70)

DEBUG: CEK HASIL SIMULASI

📂 File cdump: 980 bytes

⚠️ File WARNING ditemukan:
 NOTICE metini: MODEL_ID not AWRF in data (kg,kt)           1           1
 Setting DREC(KG,KT)%WVERT=.FALSE. to use default vertical interpolation!


📝 100 baris terakhir dari MESSAGE:





                                                                                                               ,
 RANDOM  =           0,
 TLFRAC  =  0.1000000    ,
 NTURB   =           0,
 VEGHT   =  0.5000000    ,
 ZICONTROLTF     =           0,
 WINDERRTF       =           0,
 IVMAX   =          20,
 VARSIWANT       = TIMEINDXLONGLATIZAGLSIGWTLGRZSFCTEMPSAMTFOOTSHTFDMASDENSRHFRSPHUDSWFWOUTMLHTPRES


                       ,
 IDSP    =           1,
 WVERT   = T,
 PLRISE  =           1,
 AREA    =           0,
 TERRFLG =           0,
 TFMT    =           1
 /
 ------------- End Namelist configuration -------------------

  NOTICE   main: pollutant initialization flags
      Dry deposition -  T
  NOTICE   main: meteorologi

---
## Step 2: Konversi cdump ke Format Text

File `cdump` adalah file binary yang perlu dikonversi ke text agar bisa dianalisis.

**Output:** File `cdump.txt` akan terbentuk di folder working

In [16]:
import subprocess
import os
import glob

# KONFIGURASI
cdump_file = "F:/HYSPLIT/working/cdump"
con2asc_exe = "F:/HYSPLIT/exec/con2asc.exe"
working_dir = "F:/HYSPLIT/working"

print("=" * 70)
print("STEP 2: KONVERSI CDUMP KE TEXT")
print("=" * 70)
print("\n🔄 Mengkonversi file cdump ke format text...\n")

if os.path.exists(cdump_file) and os.path.exists(con2asc_exe):
    try:
        print(f"📂 File cdump: {cdump_file}")
        print(f"   Ukuran: {os.path.getsize(cdump_file)} bytes")
        
        # Jalankan konversi
        result = subprocess.run(
            [con2asc_exe, "-i" + cdump_file],
            cwd=working_dir,
            capture_output=True,
            text=True
        )
        
        print(f"\n📝 Return code: {result.returncode}")
        if result.stdout:
            print(f"📝 File yang dihasilkan con2asc:")
            print(result.stdout)
        
        # Con2asc membuat file dengan nama cdump_XXX_XX (bukan cdump.txt)
        # Cari semua file hasil konversi
        converted_files = glob.glob(os.path.join(working_dir, "cdump_*"))
        converted_files = [f for f in converted_files if not f.endswith('.txt') and os.path.isfile(f)]
        
        if len(converted_files) > 0:
            print(f"\n✓ Ditemukan {len(converted_files)} file hasil konversi:")
            for f in converted_files:
                fname = os.path.basename(f)
                size = os.path.getsize(f)
                print(f"   - {fname} ({size} bytes)")
            
            # Gabungkan semua file menjadi cdump.txt
            cdump_txt = os.path.join(working_dir, "cdump.txt")
            all_lines = []
            
            print(f"\n🔄 Menggabungkan file menjadi cdump.txt...")
            
            for conv_file in sorted(converted_files):
                with open(conv_file, 'r') as f:
                    lines = f.readlines()
                    all_lines.extend(lines)
            
            # Simpan ke cdump.txt
            with open(cdump_txt, 'w') as f:
                f.writelines(all_lines)
            
            print(f"✓ Konversi selesai!")
            print(f"  File output: cdump.txt")
            size = os.path.getsize(cdump_txt)
            print(f"  Ukuran file: {size} bytes ({size/1024:.2f} KB)")
            print(f"  Total baris: {len(all_lines)}")
            
            # Preview data
            print("\n📊 Preview data (20 baris pertama):\n")
            for i, line in enumerate(all_lines[:20]):
                print(f"{i+1:3d}: {line.rstrip()}")
            if len(all_lines) > 20:
                print(f"\n... dan {len(all_lines)-20} baris lainnya")
            
            print("\n✅ Step 2 selesai! Lanjut ke Step 3.")
            
        else:
            print("\n⚠️  Tidak ada file hasil konversi yang ditemukan!")
            print("   Cek file CON2ASC.OUT untuk detail error:")
            con2asc_out = os.path.join(working_dir, "CON2ASC.OUT")
            if os.path.exists(con2asc_out):
                with open(con2asc_out, 'r') as f:
                    print(f.read())
        
    except Exception as e:
        print(f"❌ Error saat konversi: {e}")
        import traceback
        traceback.print_exc()
        print("\nAlternatif: Gunakan tool GUI HYSPLIT untuk visualisasi")
        
elif not os.path.exists(cdump_file):
    print("❌ File cdump tidak ditemukan!")
    print("   Pastikan simulasi di Step 1 sudah berhasil dijalankan")
else:
    print("❌ Tool con2asc.exe tidak ditemukan!")
    print(f"   Cek apakah ada di: {con2asc_exe}")

STEP 2: KONVERSI CDUMP KE TEXT

🔄 Mengkonversi file cdump ke format text...

📂 File cdump: F:/HYSPLIT/working/cdump
   Ukuran: 980 bytes

📝 Return code: 0
📝 File yang dihasilkan con2asc:
 F:/HYSPLIT/working/cdump_060_01
 F:/HYSPLIT/working/cdump_060_02
 F:/HYSPLIT/working/cdump_060_03
 F:/HYSPLIT/working/cdump_060_04
 F:/HYSPLIT/working/cdump_060_05


✓ Ditemukan 5 file hasil konversi:
   - cdump_060_01 (198 bytes)
   - cdump_060_02 (297 bytes)
   - cdump_060_03 (330 bytes)
   - cdump_060_04 (462 bytes)
   - cdump_060_05 (594 bytes)

🔄 Menggabungkan file menjadi cdump.txt...
✓ Konversi selesai!
  File output: cdump.txt
  Ukuran file: 1881 bytes (1.84 KB)
  Total baris: 57

📊 Preview data (20 baris pertama):

  1: DAY HR    LAT     LON ASH 03000
  2:  60  1  -8.16  112.92  0.19E-10
  3:  60  1  -8.11  112.92  0.22E-08
  4:  60  1  -8.16  112.97  0.22E-08
  5:  60  1  -8.11  112.97  0.19E-09
  6:  60  1  -8.16  113.02  0.44E-11
  7: DAY HR    LAT     LON ASH 03000
  8:  60  2  -8.21  112

---
## Step 3: Analisis Spasial

Ekstrak data spasial dari hasil simulasi:
- **Jarak_km**: Jarak terjauh penyebaran abu
- **Luas_km2**: Luas area yang terdampak (berdasarkan grid cell unik)
- **Sudut_deg**: Arah penyebaran dominan (0°=Utara, 90°=Timur)
- **Radius_km**: Rata-rata jarak penyebaran

**PERBAIKAN v2:**
- Deduplikasi data: hanya hitung grid cell unik (bukan per time step)
- Grid area dihitung dari spacing aktual koordinat, bukan hardcoded
- Radius & arah berdasarkan cell unik, weighted by konsentrasi maksimum
- Menampilkan info per time step

In [17]:
import numpy as np
import pandas as pd
import os
from math import radians, cos, sin, asin, sqrt, atan2, degrees

def haversine(lat1, lon1, lat2, lon2):
    """Hitung jarak antara dua koordinat (dalam km)"""
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c

def calculate_bearing(lat1, lon1, lat2, lon2):
    """Hitung bearing/sudut dari titik 1 ke titik 2 (dalam derajat)"""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = sin(dlon) * cos(lat2)
    y = cos(lat1) * sin(lat2) - sin(lat1) * cos(lat2) * cos(dlon)
    bearing = atan2(x, y)
    bearing = degrees(bearing)
    return (bearing + 360) % 360

def parse_cdump_with_timestep(filepath):
    """
    Parse cdump.txt dengan menyimpan informasi time step.
    Format con2asc: header 'DAY HR LAT LON ...' diikuti data per jam.
    Returns list of dict: {day, hr, lat, lon, conc}
    """
    data_points = []
    with open(filepath, 'r') as f:
        for line in f:
            stripped = line.strip()
            if not stripped or 'DAY' in stripped:
                continue
            parts = stripped.split()
            numeric_values = []
            for part in parts:
                try:
                    numeric_values.append(float(part))
                except ValueError:
                    continue
            if len(numeric_values) >= 5:
                data_points.append({
                    'day': int(numeric_values[0]),
                    'hr': int(numeric_values[1]),
                    'lat': numeric_values[2],
                    'lon': numeric_values[3],
                    'concentration': numeric_values[4]
                })
    return data_points

def calculate_grid_area_km2(lats, lons):
    """
    Hitung area per grid cell dari spacing aktual koordinat.
    """
    unique_lats = sorted(set(lats))
    unique_lons = sorted(set(lons))
    
    if len(unique_lats) >= 2:
        dlat_deg = abs(unique_lats[1] - unique_lats[0])
    else:
        dlat_deg = 0.25  # fallback
    
    if len(unique_lons) >= 2:
        dlon_deg = abs(unique_lons[1] - unique_lons[0])
    else:
        dlon_deg = 0.25  # fallback
    
    mean_lat = np.mean(unique_lats)
    dlat_km = dlat_deg * 111.32  # 1° latitude ≈ 111.32 km
    dlon_km = dlon_deg * 111.32 * cos(radians(abs(mean_lat)))
    
    return dlat_km * dlon_km, dlat_deg, dlon_deg

def parse_cdump_and_analyze(source_lat=-8.108, source_lon=112.922, threshold_conc=1e-12):
    """Analisis file cdump.txt dengan deduplikasi dan grid area yang benar"""
    cdump_txt = "F:/HYSPLIT/working/cdump.txt"

    if not os.path.exists(cdump_txt):
        print("❌ File cdump.txt tidak ada!")
        print("   Jalankan Step 2 terlebih dahulu.\n")
        return None, None

    print("=" * 70)
    print("STEP 3: ANALISIS SPASIAL (v2 - FIXED)")
    print("=" * 70)
    print(f"\nSumber Erupsi: {source_lat:.4f}°N, {source_lon:.4f}°E")
    print(f"Threshold konsentrasi: {threshold_conc} mg/m³")

    # === Parse data dengan time step ===
    raw_points = parse_cdump_with_timestep(cdump_txt)
    print(f"\nTotal baris data mentah: {len(raw_points)}")
    
    if len(raw_points) == 0:
        print("⚠️  Tidak ada data dalam cdump.txt!")
        return None, None
    
    # Hitung jumlah time step
    time_steps = set()
    for p in raw_points:
        time_steps.add((p['day'], p['hr']))
    n_timesteps = len(time_steps)
    sorted_ts = sorted(time_steps)
    print(f"Jumlah time step: {n_timesteps}")
    print(f"Rentang waktu: Day {sorted_ts[0][0]} Hr {sorted_ts[0][1]} → Day {sorted_ts[-1][0]} Hr {sorted_ts[-1][1]}")
    
    # === Filter data valid ===
    valid_points = []
    for p in raw_points:
        lat, lon, conc = p['lat'], p['lon'], p['concentration']
        if -90 <= lat <= 90 and -180 <= lon <= 180 and conc > threshold_conc:
            valid_points.append(p)
    
    print(f"Data valid (> threshold): {len(valid_points)}")
    
    if len(valid_points) == 0:
        print("⚠️  Tidak ada data konsentrasi yang melebihi threshold!")
        return None, None
    
    df_all = pd.DataFrame(valid_points)
    
    # === DEDUPLIKASI: Ambil konsentrasi MAKSIMUM per grid cell ===
    df_unique = df_all.groupby(['lat', 'lon']).agg(
        max_conc=('concentration', 'max'),
        mean_conc=('concentration', 'mean'),
        n_timesteps=('concentration', 'count')
    ).reset_index()
    
    n_unique = len(df_unique)
    print(f"\n🔑 Grid cell UNIK: {n_unique} (dari {len(valid_points)} data mentah)")
    print(f"   → Rasio duplikat: {len(valid_points)/n_unique:.1f}x (karena {n_timesteps} time step)")
    
    # === Hitung grid area dari spacing aktual ===
    grid_area_km2, dlat_deg, dlon_deg = calculate_grid_area_km2(
        df_unique['lat'].values, df_unique['lon'].values
    )
    print(f"\n📐 Grid spacing terdeteksi: {dlat_deg}° × {dlon_deg}°")
    print(f"   Area per cell: {grid_area_km2:.1f} km²")
    
    # === Hitung metrik spasial dari cell unik ===
    max_distance = 0
    max_lat, max_lon = source_lat, source_lon
    
    distances = []
    bearings = []
    
    for _, row in df_unique.iterrows():
        lat, lon = row['lat'], row['lon']
        distance = haversine(source_lat, source_lon, lat, lon)
        bearing = calculate_bearing(source_lat, source_lon, lat, lon)
        
        distances.append(distance)
        bearings.append(bearing)
        
        if distance > max_distance:
            max_distance = distance
            max_lat, max_lon = lat, lon
    
    df_unique['distance_km'] = distances
    df_unique['bearing_deg'] = bearings
    
    jarak_km = max_distance
    radius_km = df_unique['distance_km'].mean()
    
    # Arah dominan (circular mean dari bearing cell unik)
    rad_bearings = [radians(b) for b in bearings]
    mean_sin = np.mean([sin(r) for r in rad_bearings])
    mean_cos = np.mean([cos(r) for r in rad_bearings])
    sudut_deg = degrees(atan2(mean_sin, mean_cos)) % 360
    
    # Luas berdasarkan cell unik
    luas_km2 = n_unique * grid_area_km2
    
    # Arah dalam bahasa
    arah_labels = [
        (337.5, 22.5, "Utara"), (22.5, 67.5, "Timur Laut"),
        (67.5, 112.5, "Timur"), (112.5, 157.5, "Tenggara"),
        (157.5, 202.5, "Selatan"), (202.5, 247.5, "Barat Daya"),
        (247.5, 292.5, "Barat"), (292.5, 337.5, "Barat Laut")
    ]
    arah = "Utara"
    for lo, hi, label in arah_labels:
        if lo > hi:  # wraps around 0
            if sudut_deg >= lo or sudut_deg < hi:
                arah = label
                break
        elif lo <= sudut_deg < hi:
            arah = label
            break
    
    # === PRINT HASIL ===
    print("\n" + "=" * 70)
    print("✅ HASIL ANALISIS (CORRECTED)")
    print("=" * 70)
    
    print(f"\n📏 Jarak_km:     {jarak_km:.2f} km")
    print(f"   (Titik terjauh: {max_lat:.4f}°, {max_lon:.4f}°)")
    
    print(f"\n📐 Sudut_deg:    {sudut_deg:.2f}°")
    print(f"   (Arah dominan: {arah})")
    
    print(f"\n⭕ Radius_km:    {radius_km:.2f} km")
    print("   (Rata-rata jarak cell unik dari sumber)")
    
    print(f"\n🗺️  Luas_km2:     {luas_km2:.2f} km²")
    print(f"   ({n_unique} cell unik × {grid_area_km2:.1f} km²/cell)")
    
    print(f"\n📍 Grid cell terdampak: {n_unique}")
    print(f"   Grid spacing: {dlat_deg}° × {dlon_deg}°")
    
    # === Peringatan grid kasar ===
    if dlat_deg >= 0.1:
        print(f"\n⚠️  PERINGATAN: Grid {dlat_deg}° terlalu kasar!")
        print(f"   Setiap cell = {grid_area_km2:.0f} km² → estimasi luas tidak presisi.")
        print(f"   Saran: Ubah grid spacing di Step 1 ke 0.05° untuk resolusi ~5.5 km")
    
    print("=" * 70)

    # === Buat summary DataFrame ===
    summary = pd.DataFrame({
        'Metrik': ['Jarak_km', 'Luas_km2', 'Sudut_deg', 'Radius_km'],
        'Nilai': [jarak_km, luas_km2, sudut_deg, radius_km],
        'Keterangan': [
            f'Jarak terjauh: {max_lat:.4f}°, {max_lon:.4f}°',
            f'{n_unique} cell unik × {grid_area_km2:.1f} km²/cell (grid {dlat_deg}°)',
            f'Arah dominan: {arah}',
            f'Rata-rata jarak {n_unique} cell unik dari sumber'
        ]
    })

    print("\n📋 TABEL RINGKASAN:\n")
    print(summary.to_string(index=False))
    
    # Detail per cell (bukan per time step)
    detail = df_unique[['lat', 'lon', 'max_conc', 'mean_conc', 'n_timesteps', 
                         'distance_km', 'bearing_deg']].copy()
    detail = detail.sort_values('distance_km', ascending=False).reset_index(drop=True)
    
    print("\n📊 TOP 10 Cell Terjauh:")
    print(detail.head(10).to_string(index=False))
    
    print("\n✅ Step 3 selesai! Lanjut ke Step 4.\n")

    return summary, detail

# ========================================================================
# JALANKAN ANALISIS MENGGUNAKAN PARAMETER DARI CELL KONFIGURASI
# ========================================================================

# Cek apakah parameter konfigurasi sudah didefinisikan
if 'LAT_SUMBER' not in locals():
    print("\n⚠️  BELUM MENJALANKAN CELL KONFIGURASI!")
    print("   Jalankan cell 'PARAMETER KONFIGURASI' terlebih dahulu.\n")
    result = None, None
else:
    # Gunakan parameter dari cell konfigurasi
    result = parse_cdump_and_analyze(
        source_lat=LAT_SUMBER,
        source_lon=LON_SUMBER,
        threshold_conc=THRESHOLD_CONC
    )

if result and result[0] is not None:
    summary_df, detail_df = result
    print("✅ Data siap untuk di-export!")

STEP 3: ANALISIS SPASIAL (v2 - FIXED)

Sumber Erupsi: -8.1080°N, 112.9220°E
Threshold konsentrasi: 1e-10 mg/m³

Total baris data mentah: 52
Jumlah time step: 5
Rentang waktu: Day 60 Hr 1 → Day 60 Hr 5
Data valid (> threshold): 36

🔑 Grid cell UNIK: 14 (dari 36 data mentah)
   → Rasio duplikat: 2.6x (karena 5 time step)

📐 Grid spacing terdeteksi: 0.049999999999998934° × 0.04999999999999716°
   Area per cell: 30.7 km²

✅ HASIL ANALISIS (CORRECTED)

📏 Jarak_km:     35.50 km
   (Titik terjauh: -8.3600°, 113.1200°)

📐 Sudut_deg:    139.26°
   (Arah dominan: Tenggara)

⭕ Radius_km:    19.41 km
   (Rata-rata jarak cell unik dari sumber)

🗺️  Luas_km2:     429.25 km²
   (14 cell unik × 30.7 km²/cell)

📍 Grid cell terdampak: 14
   Grid spacing: 0.049999999999998934° × 0.04999999999999716°

📋 TABEL RINGKASAN:

   Metrik      Nilai                                                Keterangan
 Jarak_km  35.496065                        Jarak terjauh: -8.3600°, 113.1200°
 Luas_km2 429.252811 14 cell 

---
## Step 4: Export Hasil ke CSV/Excel

Simpan hasil analisis ke file:
- **analisis_spasial_summary.csv**: Ringkasan 4 metrik utama
- **analisis_spasial_detail.csv**: Detail semua titik yang terdampak
- **analisis_dispersi_abu.xlsx**: File Excel dengan kedua sheet

In [18]:
import pandas as pd
import os
import subprocess

def export_results(summary_df, detail_df, output_dir="F:/HYSPLIT/working"):
    """
    Export hasil analisis ke file CSV dan Excel
    """
    
    if summary_df is None:
        print("❌ Tidak ada data untuk di-export!")
        print("   Pastikan Step 3 sudah berhasil dijalankan.\n")
        return
    
    print("=" * 70)
    print("STEP 4: EXPORT HASIL")
    print("=" * 70)
    print("\n💾 Menyimpan hasil analisis...\n")
    
    # Pastikan folder output ada
    os.makedirs(output_dir, exist_ok=True)
    
    # File output
    summary_csv = os.path.join(output_dir, "analisis_spasial_summary.csv")
    detail_csv = os.path.join(output_dir, "analisis_spasial_detail.csv")
    excel_file = os.path.join(output_dir, "analisis_dispersi_abu.xlsx")
    
    # Export ke CSV
    summary_df.to_csv(summary_csv, index=False, encoding='utf-8-sig')
    detail_df.to_csv(detail_csv, index=False, encoding='utf-8-sig')
    
    print("✅ File berhasil dibuat!\n")
    print(f"📄 Summary CSV: {summary_csv}")
    print(f"📄 Detail CSV:  {detail_csv}")
    
    # Export ke Excel (jika openpyxl tersedia)
    try:
        with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
            summary_df.to_excel(writer, sheet_name='Summary', index=False)
            detail_df.to_excel(writer, sheet_name='Detail Data', index=False)
        
        print(f"📊 Excel File:  {excel_file}")
    except ImportError:
        print("⚠️  Library openpyxl tidak tersedia, skip export Excel")
        print("   Install dengan: pip install openpyxl")
    
    print("\n" + "=" * 70)
    print("✅ SEMUA STEP SELESAI!")
    print("=" * 70)
    print("\nHasil analisis telah disimpan di:")
    print(f"   {output_dir}")
    # print("\n📂 Membuka folder...\n")
    
    # Buka folder hasil
    # subprocess.Popen(f'explorer "{output_dir}"')

# EXPORT HASIL
# Pastikan Step 3 sudah dijalankan dan variabel result tersedia
if 'result' in locals() and result is not None and result[0] is not None:
    summary_df, detail_df = result
    export_results(summary_df, detail_df)
else:
    print("⚠️  Jalankan Step 3 terlebih dahulu untuk mendapatkan data!")

STEP 4: EXPORT HASIL

💾 Menyimpan hasil analisis...

✅ File berhasil dibuat!

📄 Summary CSV: F:/HYSPLIT/working\analisis_spasial_summary.csv
📄 Detail CSV:  F:/HYSPLIT/working\analisis_spasial_detail.csv
📊 Excel File:  F:/HYSPLIT/working\analisis_dispersi_abu.xlsx

✅ SEMUA STEP SELESAI!

Hasil analisis telah disimpan di:
   F:/HYSPLIT/working


---
## 🎯 Ringkasan

### Cara Menggunakan Notebook:

 1. **Edit Parameter** - Buka cell "PARAMETER KONFIGURASI" dan sesuaikan:
   - `LAT_SUMBER`, `LON_SUMBER` - Lokasi erupsi
   - `TINGGI_LETUSAN_M` - Tinggi kolom abu (1000-10000 m)
   - `RUNTIME_JAM` - Durasi simulasi (3-72 jam)
   - `TIMESTAMP` - Waktu erupsi (HARUS sesuai data meteorologi!)
   - `MET_FILE` - Nama file meteorologi GDAS
   - `GRID_SPACING` - Resolusi (0.05° untuk presisi, 0.25° untuk cepat)

2. **Jalankan Semua Cell** - Dari "PARAMETER KONFIGURASI" sampai Step 4, secara berurutan

3. **Review Hasil** - Cek file CSV/Excel di `F:/HYSPLIT/working/`

---

### Output yang Dihasilkan:

1. **File cdump** - Hasil simulasi HYSPLIT (binary)
2. **File cdump.txt** - Hasil simulasi dalam format text
3. **analisis_spasial_summary.csv** - Ringkasan metrik:
   - **Jarak_km**: Jarak terjauh penyebaran abu dari sumber
   - **Luas_km2**: Estimasi luas area terdampak
   - **Sudut_deg**: Arah penyebaran dominan (0°=Utara, 90°=Timur, 180°=Selatan, 270°=Barat)
   - **Radius_km**: Rata-rata jarak penyebaran
4. **analisis_spasial_detail.csv** - Detail semua grid cell terdampak (lat, lon, konsentrasi, jarak)
5. **analisis_dispersi_abu.xlsx** - File Excel lengkap dengan kedua sheet

---